In [3]:
pip install pypdf

In [4]:
from typing import Annotated

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from typing_extensions import TypedDict


# ----------------------------------------------------
# Load Documents
# ----------------------------------------------------

loader = PyPDFLoader("Class 11 -2nd Holy Qurbana – A Study English.pdf")

docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

documents = splitter.split_documents(docs)


# ----------------------------------------------------
# Embeddings + Vector Store
# ----------------------------------------------------

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

retriever = vectorstore.as_retriever()


# ----------------------------------------------------
# Retriever Tool
# ----------------------------------------------------

@tool
def retrieve(query: str):
    """
    Retrieve relevant documents from the knowledge base.
    """

    docs = retriever.invoke(query)

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


tools = [retrieve]


# ----------------------------------------------------
# LLM
# ----------------------------------------------------

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0
)

llm = llm.bind_tools(tools)


# ----------------------------------------------------
# State
# ----------------------------------------------------

class State(TypedDict):
    messages: Annotated[list, add_messages]


# ----------------------------------------------------
# Agent Node
# ----------------------------------------------------

def chatbot(state: State):

    response = llm.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }


# ----------------------------------------------------
# Graph
# ----------------------------------------------------

builder = StateGraph(State)

builder.add_node(
    "chatbot",
    chatbot
)

builder.add_node(
    "tools",
    ToolNode(tools)
)

builder.add_edge(
    START,
    "chatbot"
)

builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

builder.add_edge(
    "tools",
    "chatbot"
)

graph = builder.compile()


# ----------------------------------------------------
# Run
# ----------------------------------------------------

while True:

    question = input("You : ")

    if question.lower() in ["exit", "quit"]:
        break

    result = graph.invoke(
        {
            "messages": [
                HumanMessage(content=question)
            ]
        }
    )

    print(
        "\nAI :",
        result["messages"][-1].content
    )


AI : **The Importance of the Passover Feast**

The Passover feast is a significant event in the Jewish calendar, commemorating the Israelites' liberation from slavery in Egypt. It is observed for seven days, starting on the 15th day of the Hebrew month of Nisan.

**Historical Significance**

The Passover feast marks the beginning of the Jewish holiday season, known as Pesach. According to the biblical account in Exodus, the Israelites were forced into slavery by the Egyptians and were subjected to harsh treatment. God heard their cries for help and sent Moses to lead them out of slavery.

**Spiritual Significance**

The Passover feast is a celebration of faith, redemption, and freedom. It represents the Israelites' deliverance from bondage and their commitment to follow God's laws. The feast also symbolizes the sacrifice of Jesus Christ, who was crucified on the eve of Passover (Matthew 26:17-30).

**Traditions and Customs**

The Passover feast involves several traditions and customs: